In [1]:
import pandas as pd
import numpy  as np
import seaborn as sns
import matplotlib.pyplot as plt
import sqlite3

# Loading the Data

In [2]:
excel_file = "team_performance_data_fixed.xlsx"
full_data = pd.read_excel(excel_file, sheet_name=None)

df_manager = full_data['dim_manager']
df_employee = full_data['dim_employee']
df_case = full_data['fact_case']
df_paused_case = full_data['fact_paused_case']


Inspecting the loaded Data

In [3]:
print("Manager Table: ")
df_manager.info()
print("\nEmployee Table: ")
df_employee.info()
print("\nCase Table: ")
df_case.info()
print("\nPaused case Table: ")
df_paused_case.info()

Manager Table: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   Manager_ID                10 non-null     object        
 1   Manager_Name              10 non-null     object        
 2   Team_Name                 10 non-null     object        
 3   Manager_Hire_Date         10 non-null     datetime64[ns]
 4   Manager_Experience_Years  10 non-null     float64       
dtypes: datetime64[ns](1), float64(1), object(3)
memory usage: 532.0+ bytes

Employee Table: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Employee_ID    100 non-null    object        
 1   Employee_Name  100 non-null    object        
 2   Manager_ID     100 non-null    object      

In [4]:
df_manager.head()

,Manager_ID,Manager_Name,Team_Name,Manager_Hire_Date,Manager_Experience_Years
0,MGR001,Manager 1,Team 1,2018-01-01,6.5
1,MGR002,Manager 2,Team 2,2018-06-30,7.0
2,MGR003,Manager 3,Team 3,2018-12-27,7.5
3,MGR004,Manager 4,Team 4,2019-06-25,8.0
4,MGR005,Manager 5,Team 5,2019-12-22,8.5


In [5]:
df_employee.head()

,Employee_ID,Employee_Name,Manager_ID,Hire_Date,Shift
0,EMP0001,Employee 1,MGR001,2021-01-28,Morning
1,EMP0002,Employee 2,MGR001,2024-05-13,Morning
2,EMP0003,Employee 3,MGR001,2023-12-04,Morning
3,EMP0004,Employee 4,MGR001,2021-04-07,Morning
4,EMP0005,Employee 5,MGR001,2025-06-25,Evening


In [6]:
df_case.head()

,Case_ID,Case_Owner_ID,Customer_ID,Case_Status,Category,Channel,Created_Time,First_Response_Time,Escalation_Time,Closed_Time,Outbound,Reopen_Count,CSAT_Score
0,CASE000001,EMP0086,CUST04446,Closed,Billing,Chat,2026-06-01 21:49:00,2026-06-02 00:30:00,NaT,2026-06-02 03:06:28.229,No,0,5
1,CASE000002,EMP0018,CUST04331,Closed,Payment,Email,2026-06-16 01:19:00,2026-06-16 01:48:00,2026-06-16 02:41:00,2026-06-17 14:04:18.268,No,0,5
2,CASE000003,EMP0003,CUST01245,Closed,General Query,Chat,2026-02-20 16:23:00,2026-02-20 16:24:00,2026-02-20 17:28:00,2026-02-21 01:54:38.874,No,0,1
3,CASE000004,EMP0064,CUST03578,Closed,Billing,Chat,2026-05-25 19:57:00,2026-05-25 19:59:00,NaT,2026-05-25 22:35:37.986,Yes,2,2
4,CASE000005,EMP0037,CUST02446,Closed,Access,Chat,2026-06-05 14:29:00,2026-06-05 15:18:00,2026-06-05 18:12:00,2026-06-06 01:11:34.632,Yes,2,3


In [7]:
df_paused_case.head()

,Pause_ID,Case_ID,Case_Owner_ID,Pause_Start_Time,Resume_Time,Resumed_By,Agent_Response_Time,Pause_Reason
0,CASE002657-P01,CASE002657,EMP0096,2026-05-02 15:09:00,2026-05-03 01:27:00,Customer,2026-05-03 02:34:00,System Dependency
1,CASE002657-P02,CASE002657,EMP0096,2026-05-02 16:39:00,2026-05-02 18:05:00,Customer,2026-05-02 19:30:00,System Dependency
2,CASE000446-P01,CASE000446,EMP0014,2026-01-29 05:22:00,2026-01-29 23:49:00,Customer,2026-01-30 01:07:00,System Dependency
3,CASE009506-P01,CASE009506,EMP0003,2026-03-21 10:49:00,2026-03-21 12:58:00,Customer,2026-03-21 13:32:00,System Dependency
4,CASE000333-P01,CASE000333,EMP0013,2026-04-13 07:10:00,2026-04-13 15:42:00,Customer,2026-04-13 16:33:00,Waiting for Internal Team


# Data Quality Validation

In [8]:
# checking for duplicates in manager table

# shows the list of all unique values can be used for data with small number of entries
# df_manager["Manager_ID"].nunique 

# Checking for duplicated values
df_manager["Manager_ID"].duplicated().sum() 

np.int64(0)

verifying if all the entries are unique in the Manager, Employee and the Case table.

In [9]:
df_manager["Manager_ID"].nunique() == df_manager["Manager_ID"].count() 

np.True_

In [10]:
df_employee["Employee_ID"].nunique() == df_employee["Employee_ID"].count() 

np.True_

In [11]:
df_case["Case_ID"].nunique() == df_case["Case_ID"].count() 

np.True_

# Validating Table Relationships

In [12]:
# Checking for INvalid Managers

invalid_managers = df_employee[
    ~df_employee["Manager_ID"].isin(df_manager["Manager_ID"])
]

len(invalid_managers)

0

In [13]:
invalid_owners = df_case[
    ~df_case["Case_Owner_ID"].isin(df_employee["Employee_ID"])
]

len(invalid_owners)

0

# Validating timestamp sequence

In [14]:
# First Response time is always after the creation time
invalid_frt = df_case[
    df_case["First_Response_Time"] < df_case["Created_Time"]
]
invalid_frt.shape

(0, 13)

In [15]:
# Closed time is always after the creation time
invalid_closed = df_case[
    df_case["Closed_Time"] < df_case["Created_Time"]
]
invalid_closed.shape

(0, 13)

In [16]:
# Resume time is always after the pause_start time
invalid_pause = df_paused_case[
    df_paused_case["Resume_Time"] < df_paused_case["Pause_Start_Time"]
]
invalid_pause.shape

(246, 8)

# chaecking for any missing Values

In [17]:
df_case.isnull().sum()

Case_ID                   0
Case_Owner_ID             0
Customer_ID               0
Case_Status               0
Category                  0
Channel                   0
Created_Time              0
First_Response_Time       0
Escalation_Time        5922
Closed_Time               0
Outbound                  0
Reopen_Count              0
CSAT_Score                0
dtype: int64

# Feature Engineering 
Creating analytics table with new features to carry on the Data analysis. 

**Features Created :** 
Case resolution time = Case Closing time - Case creation time   
First response Delay = First Response time - Case Creation time  
Escalation Submit = Escalation time - Case Creation time
NECRT Flag = Determine if case resolution time for non escalated cases is < 360 mins ( 6 hours) {Non escalated cases closed within 6 hours}
Triage Flag = Determine if escalation submit time for escalated cases is < 90 mins ( 1.5 hours) {Escalation report was submitted within 90 minutes}



In [18]:
# creating analysis data frame to store the deduced columns for each case.  
a_df = pd.DataFrame()

# adding columns in the data frame
a_df["Case_ID"] = df_case["Case_ID"]

a_df["Case_Owner"] = df_case["Case_Owner_ID"]

# Calculating First response Time
a_df["FRD_sec"] = (
    df_case["First_Response_Time"]
    - df_case["Created_Time"]
).dt.total_seconds()

# Calculating Case resolution time
a_df["CRT_min"] = (
    df_case["Closed_Time"]
    - df_case["Created_Time"]
).dt.total_seconds()/60

# Creating Outbound column
a_df["Outbound"] = df_case["Outbound"]

#Calculating Escalation Time
a_df["Escalation_submit_min"] = round((df_case["Escalation_Time"]- df_case["Created_Time"]
).dt.total_seconds()/60, 0)

# Creating flag for Triage 
# defining conditions to determine the value for the triage column
conditions = [
    a_df["Escalation_submit_min"].isna(),  # If Yes
    a_df["Escalation_submit_min"]< 91  # If No to Condition 1, but Yes to Condition 2
]

# Define the matching values for each condition
choices = [None,"Yes"]

# Create the new column (default covers "If No to both")
a_df["Triage"] = np.select(conditions, choices, default="No")

# Creating flag for NECRT 
conditions = [
    a_df["Escalation_submit_min"].notna(),  # If Yes
    a_df["CRT_min"] < 360  # If No to Condition 1, but Yes to Condition 2
]

# Define the matching values for each condition
choices = [None,"Yes"]

# Create the new column (default covers "If No to both")
a_df["NECRT"] = np.select(conditions, choices, default="No")

a_df["CSAT"] = np.where( df_case["CSAT_Score"]>3 , "Yes", "No")




In [19]:
# merging with the employee table to fetch the manager details in the result data frame

a_df = a_df.merge(
    df_employee[['Employee_ID', 'Manager_ID']], 
    left_on='Case_Owner',          
    right_on='Employee_ID', 
    how='left'
)

In [20]:
a_df = a_df.drop('Employee_ID', axis=1)

In [21]:
a_df.head(50)

,Case_ID,Case_Owner,FRD_sec,CRT_min,Outbound,Escalation_submit_min,Triage,NECRT,CSAT,Manager_ID
0,CASE000001,EMP0086,9660.0,317.470483,No,NaN,None,Yes,Yes,MGR009
1,CASE000002,EMP0018,1740.0,2205.304467,No,82.0,Yes,None,Yes,MGR002
2,CASE000003,EMP0003,60.0,571.647900,No,65.0,Yes,None,No,MGR001
3,CASE000004,EMP0064,120.0,158.633100,Yes,NaN,None,Yes,No,MGR007
4,CASE000005,EMP0037,2940.0,642.577200,Yes,223.0,No,None,No,MGR004
5,CASE000006,EMP0047,2820.0,103.626217,Yes,NaN,None,Yes,Yes,MGR005
6,CASE000007,EMP0008,1200.0,2866.254083,Yes,21.0,Yes,None,No,MGR001
7,CASE000008,EMP0038,1920.0,526.490633,Yes,29.0,Yes,None,Yes,MGR004
8,CASE000009,EMP0065,1860.0,2923.083717,No,NaN,None,No,No,MGR007
9,CASE000010,EMP0036,2340.0,403.799033,No,NaN,None,No,No,MGR004


# Defining KPIs

 **FRT%**: Percentage of cases in which the first response from the agent was given within SLA. (3 minutes). 
 **Outbound%**: Percentage of total closed cases in which an outbound call was made.
 **Triage%**: Percentage of escalated cases in which the escalation report was made within SLA (90 minutes). 
 **NECRT%**: Percentage of non escalated cases that were closed within the SLA (5 Hours). 
 **CSAT%**: Percentage of cases in which the customer has rated the agent above 3 in the scale of 1-5. 
 **RR%**: Percentage of paused cases in which the response was given within SLA when the case was reopened. 
 **CRT**: Average time taken by the agent to close the cases. (Case resolution time) 
 


# Preapring Analysis table to analyse the closed cases data
1. CSAT %
2. NECRT %
3. Triage %
4. Outbound %
5. Average CRT (minutes)

In [22]:
# creating result data frame to store the result for each team. 
r_df = pd.DataFrame()
r_df['Manager_ID'] = a_df['Manager_ID']

In [23]:
# calculating KPIs

# CSAT percentage
r_df["CSAT_score"] = (
    a_df.groupby("Manager_ID")["CSAT"]
    .transform(lambda x: (x == "Yes").mean() * 100)
)

case_counts = a_df.groupby("Manager_ID")["CSAT"].count()
r_df["Case_count"] = r_df["Manager_ID"].map(case_counts)

# NECRT
r_df["NECRT_%"] = (
    a_df.groupby("Manager_ID")["NECRT"]
    .transform(lambda x: (x == "Yes").mean() * 100)
)

# Triage %
triage = a_df.groupby("Manager_ID")["Triage"].apply(lambda x: (x == "Yes").mean() * 100)
r_df["Triage_%"] = r_df["Manager_ID"].map(triage)

# Outbound %
outbound = a_df.groupby("Manager_ID")["Outbound"].apply(lambda x: (x == "Yes").mean() * 100)
r_df["Outbound_%"] = r_df["Manager_ID"].map(outbound)

# Average CRT

avg_crt = a_df.groupby("Manager_ID")["CRT_min"].mean()
# 2. Map the averages directly to the matching Manager_ID rows in r_df
r_df["Avg_CRT(min)"] = r_df["Manager_ID"].map(avg_crt)




r_df = round(r_df, 2)

In [24]:
r_df.head()

,Manager_ID,CSAT_score,Case_count,NECRT_%,Triage_%,Outbound_%,Avg_CRT(min)
0,MGR009,58.75,989,44.89,29.52,34.48,1241.64
1,MGR002,60.80,977,46.26,31.32,36.13,1156.21
2,MGR001,58.83,991,43.29,31.89,34.71,1274.59
3,MGR007,61.07,1025,45.27,31.71,32.88,1240.45
4,MGR004,61.40,1000,47.10,28.80,37.30,1152.69


# Preparation for backlog analysis

In [25]:
df_paused_case.head()

,Pause_ID,Case_ID,Case_Owner_ID,Pause_Start_Time,Resume_Time,Resumed_By,Agent_Response_Time,Pause_Reason
0,CASE002657-P01,CASE002657,EMP0096,2026-05-02 15:09:00,2026-05-03 01:27:00,Customer,2026-05-03 02:34:00,System Dependency
1,CASE002657-P02,CASE002657,EMP0096,2026-05-02 16:39:00,2026-05-02 18:05:00,Customer,2026-05-02 19:30:00,System Dependency
2,CASE000446-P01,CASE000446,EMP0014,2026-01-29 05:22:00,2026-01-29 23:49:00,Customer,2026-01-30 01:07:00,System Dependency
3,CASE009506-P01,CASE009506,EMP0003,2026-03-21 10:49:00,2026-03-21 12:58:00,Customer,2026-03-21 13:32:00,System Dependency
4,CASE000333-P01,CASE000333,EMP0013,2026-04-13 07:10:00,2026-04-13 15:42:00,Customer,2026-04-13 16:33:00,Waiting for Internal Team


In [26]:
# creating backlog data frame for analysis


adf_backlog = pd.DataFrame()

# Copying necessary data for the analysis table 
adf_backlog['Pause_ID'] = df_paused_case['Pause_ID']
adf_backlog['Case_ID'] = df_paused_case['Case_ID']
adf_backlog['Case_Owner_ID'] = df_paused_case['Case_Owner_ID']
adf_backlog['agent_resp_delay'] = (df_paused_case['Agent_Response_Time'] - df_paused_case['Resume_Time'])
adf_backlog['RR_Sla'] = np.where( adf_backlog['agent_resp_delay'].dt.total_seconds() > 3600, 'No', 'Yes')


# Adding the manager column to in the analysis table
adf_backlog = adf_backlog.merge(
    df_employee[['Employee_ID', 'Manager_ID']], 
    left_on='Case_Owner_ID',          
    right_on='Employee_ID', 
    how='left'
)

adf_backlog = adf_backlog.drop(['Employee_ID','agent_resp_delay'] , axis=1)


adf_backlog.head()

,Pause_ID,Case_ID,Case_Owner_ID,RR_Sla,Manager_ID
0,CASE002657-P01,CASE002657,EMP0096,No,MGR010
1,CASE002657-P02,CASE002657,EMP0096,No,MGR010
2,CASE000446-P01,CASE000446,EMP0014,No,MGR002
3,CASE009506-P01,CASE009506,EMP0003,Yes,MGR001
4,CASE000333-P01,CASE000333,EMP0013,Yes,MGR002


In [27]:
# Using the analysis table to generate the backlog reports. 
# 1. Backlog related KPIs : Backlog percentage(percentage distribution of total backlog cases per team) ,
# 2. RR percentage( percentage of paused cased that were responded within SLA time when resumed) 

In [28]:
rdf_backlog = pd.DataFrame()
rdf_backlog['Manager_ID'] = adf_backlog['Manager_ID'].unique()

rdf_backlog["RR_%"] = (
    adf_backlog.groupby("Manager_ID")["RR_Sla"]
    .transform(lambda x: (x == "Yes").mean() * 100)
)
backlog_prct = adf_backlog['Manager_ID'].value_counts(normalize=True) * 100

# 2. Map those percentages to the Manager_ID column in your target DataFrame
rdf_backlog['Backlog_%'] = rdf_backlog['Manager_ID'].map(backlog_prct)


rdf_backlog

,Manager_ID,RR_%,Backlog_%
0,MGR010,43.362832,9.676729
1,MGR002,43.362832,9.505459
2,MGR001,49.774775,9.997859
3,MGR003,49.892934,9.655320
4,MGR005,49.774775,10.597303
5,MGR007,48.780488,9.441233
6,MGR008,46.868687,10.939842
7,MGR004,52.380952,9.976450
8,MGR006,52.380952,10.104903
9,MGR009,47.162427,10.104903


# Generating final report by combining the backlog and closed cases report. 

In [29]:
# creating data frame for final report

f_df = pd.DataFrame()
f_df = r_df.merge(rdf_backlog, on = 'Manager_ID')

f_df.head()

,Manager_ID,CSAT_score,Case_count,NECRT_%,Triage_%,Outbound_%,Avg_CRT(min),RR_%,Backlog_%
0,MGR009,58.75,989,44.89,29.52,34.48,1241.64,47.162427,10.104903
1,MGR002,60.80,977,46.26,31.32,36.13,1156.21,43.362832,9.505459
2,MGR001,58.83,991,43.29,31.89,34.71,1274.59,49.774775,9.997859
3,MGR007,61.07,1025,45.27,31.71,32.88,1240.45,48.780488,9.441233
4,MGR004,61.40,1000,47.10,28.80,37.30,1152.69,52.380952,9.976450


In [30]:
# exporting the results

f_df.to_csv('result_file.csv', index=False)